In [2]:
"""
재료명 매핑 및 avg_price 업데이트 스크립트
- 가격 없는 재료를 66개 기준 재료에 매핑
- ingredients.avg_price 업데이트
"""

import mysql.connector
from mysql.connector import Error
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("map_prices.log", encoding="utf-8")
    ]
)
log = logging.getLogger(__name__)

DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "database": "cooking_db",
    "user": "root",
    "password": "root",
    "charset": "utf8mb4"
}

# 매핑 딕셔너리: {매핑할 재료명 키워드: 기준 재료명}
# 기준 재료명은 반드시 ingredients 테이블에 avg_price가 있는 것
PRICE_MAPPING = {
    # 계란류
    "달걀": "계란",
    "메추리알": "계란",
    "달걀노른자": "계란",
    "달걀흰자": "계란",
    "달걀 노른자": "계란",
    "달걀 흰자": "계란",
    "달걀물": "계란",
    "달걀지단": "계란",
    "달걀프라이": "계란",
    "삶은 달걀": "계란",
    "삶은달걀": "계란",
    "깐 메추리알": "계란",

    # 돼지고기류
    "삼겹살": "돼지고기",
    "대패삼겹살": "돼지고기",
    "대패 삼겹살": "돼지고기",
    "냉동삼겹살": "돼지고기",
    "돼지 목살": "돼지고기",
    "돼지갈비": "돼지고기",
    "돼지고기 목살": "돼지고기",
    "돼지고기 등심": "돼지고기",
    "돼지고기 앞다리살": "돼지고기",
    "돼지고기 뒷다리살": "돼지고기",
    "돼지고기 후지": "돼지고기",
    "돼지등갈비": "돼지고기",
    "등갈비": "돼지고기",
    "항정살": "돼지고기",
    "부채살": "돼지고기",
    "목심다짐육": "돼지고기",
    "간 돼지고기": "돼지고기",
    "간돼지고기": "돼지고기",
    "다진 돼지고기": "돼지고기",
    "뒷다리살": "돼지고기",
    "돼지고기전지": "돼지고기",
    "돼지지방": "돼지고기",

    # 소고기류
    "소갈비": "소고기(국산)",
    "LA갈비": "소고기(국산)",
    "갈비": "소고기(국산)",
    "불고기용소고기": "소고기(국산)",
    "소 불고기용": "소고기(국산)",
    "소고기 불고기용": "소고기(국산)",
    "소고기목심": "소고기(국산)",
    "소고기슬라이스": "소고기(국산)",
    "쇠고기등심": "소고기(국산)",
    "안심": "소고기(국산)",
    "차돌박이": "소고기(국산)",
    "소 양지": "소고기(국산)",
    "소양지": "소고기(국산)",
    "우목심": "소고기(국산)",
    "간소고기": "소고기(국산)",
    "다진소고기": "소고기(국산)",
    "양지다짐육": "소고기(국산)",

    # 닭고기류
    "닭": "닭고기",
    "닭가슴살": "닭고기",
    "닭 다리살": "닭고기",
    "닭다리살": "닭고기",
    "닭봉": "닭고기",
    "닭 염지": "닭고기",
    "토막 닭": "닭고기",
    "토막닭": "닭고기",
    "토막 닭고기": "닭고기",
    "삶은 닭": "닭고기",
    "닭근위": "닭고기",
    "닭날": "닭고기",

    # 마늘류
    "마늘": "깐마늘",
    "다진마늘": "깐마늘",
    "통마늘": "깐마늘",
    "마늘 쪽": "깐마늘",
    "마늘쫑": "깐마늘",

    # 고추류
    "고추": "청양고추",
    "풋고추": "청양고추",
    "오이고추": "청양고추",
    "홍고추": "붉은고추",
    "건고추": "붉은고추",
    "건 고추": "붉은고추",
    "고추기름": "붉은고추",
    "불린 고춧가루": "고춧가루",
    "고운 고춧가루": "고춧가루",
    "고운고추가루": "고춧가루",
    "굵은 고춧가루": "고춧가루",
    "굵은고추가루": "고춧가루",

    # 파류
    "쪽파": "대파",
    "파 채": "대파",
    "파채": "대파",
    "대파 흰 부분": "대파",
    "대파 흰부분": "대파",
    "대파흰부분": "대파",

    # 버섯류
    "느타리버섯": "버섯",
    "표고버섯": "버섯",
    "표고 버섯": "버섯",
    "팽이버섯": "버섯",
    "새송이": "버섯",
    "새송이버섯": "버섯",
    "양송이": "버섯",
    "양송이버섯": "버섯",
    "목이버섯": "버섯",
    "건 목이버섯": "버섯",

    # 쌀/면류
    "소면": "국수",
    "가락국수": "국수",
    "가락국수면": "국수",
    "칼국수면": "국수",
    "메밀면": "국수",
    "중면": "국수",
    "쫄면": "국수",
    "쌀가루": "쌀",
    "찹쌀가루": "쌀",
    "습식쌀가루": "쌀",
    "쌀떡": "쌀",
    "가래떡": "쌀",
    "떡국떡": "쌀",
    "떡국 떡": "쌀",
    "불린떡국떡": "쌀",
    "밀가루떡": "밀가루",

    # 배추/무류
    "알배추": "배추",
    "알배기 배추": "배추",
    "절인배추": "배추",
    "배추 포기": "배추",
    "무우": "무",
    "무절임": "무",
    "무말랭이": "무",
    "무청": "무",

    # 양파류
    "다진 양파": "양파",
    "양파 채": "양파",

    # 두부류
    "순두부": "두부",
    "부침용 두부": "두부",

    # 감자류
    "냉동감자": "감자",
    "삶은 고구마": "고구마",

    # 호박류
    "돼지호박": "애호박",
    "돼지 호박": "애호박",
    "주키니호박": "애호박",

    # 오이류
    "오이채": "오이",
    "오이장아찌": "오이",

    # 당근류
    "당근 채": "당근",
    "당근채": "당근",

    # 양배추류
    "양배추 채": "양배추",
    "양배추채": "양배추",

    # 상추류
    "로메인": "상추",
    "양상추": "상추",
    "치커리": "상추",
    "어린잎채소": "상추",
    "쌈 채소": "상추",

    # 토마토류
    "완숙 토마토": "토마토",
    "토마토 작은거": "토마토",

    # 생강류
    "다진생강": "깐마늘",
    "한 생강": "깐마늘",

    # 새우류
    "칵테일 새우": "새우",
    "칵테일새우": "새우",
    "자숙 새우": "새우",
    "건 새우": "새우",
    "건새우": "새우",

    # 멸치류
    "국멸치": "마른멸치",
    "국물용 멸치": "마른멸치",
    "손질한 국멸치": "마른멸치",
    "중멸치": "마른멸치",
    "육수용멸치": "마른멸치",
    "훈연멸치": "마른멸치",
    "멸치가루": "마른멸치",
    "훈연멸치가루": "마른멸치",
    "멸치 액젓": "마른멸치",
    "멸치액젓": "마른멸치",

    # 오징어류
    "손질오징어": "오징어",
    "오징어채": "오징어",

    # 조개류
    "바지락": "조개",
    "홍합": "조개",
    "홍합살": "조개",

    # 설탕류
    "황설탕": "설탕",
    "올리고당": "설탕",
    "물엿": "설탕",

    # 식초류
    "사과식초": "식초",
    "양조식초": "식초",
    "양조 식초": "식초",

    # 참기름류
    "들기름": "참기름",

    # 콩나물류
    "숙주": "콩나물",

    # 라면류
    "진라면": "라면",
    "너구리": "라면",
    "짜파게티": "라면",

    # 우유류
    "생크림": "우유",
    "흰우유": "우유",

    # 빵류
    "식빵": "빵",
    "모닝빵": "빵",
    "바게트": "빵",
    "햄버거빵": "빵",
    "통식빵": "빵",
    "핫도그빵": "빵",

    # 파프리카
    "노랑파프리카": "파프리카",
    "빨간파프리카": "파프리카",
    "빨강파프리카": "파프리카",
    "청피망": "파프리카",
    "피망": "파프리카",

    # 소고기 추가
    "불고기": "소고기(국산)",
    "갈비탕": "소고기(국산)",
    "소고기": "소고기(국산)",
    "탕수육고기": "돼지고기",
    "편육": "돼지고기",
    "족발": "돼지고기",
    "돈등심": "돼지고기",
    "돼지고기뒷다리살": "돼지고기",
    "돼지고기등심": "돼지고기",
    "돼지고기전지": "돼지고기",
    "뒷다리살카레": "돼지고기",
    "돼지등갈비튀김": "돼지고기",

    # 닭고기 추가
    "치킨": "닭고기",
    "조미닭가슴살": "닭고기",
    "찜닭": "닭고기",

    # 면류 추가
    "스파게티면": "국수",
    "삶은스파게티면": "국수",
    "우동면": "국수",
    "중화제육면": "국수",
    "쫄면사리": "국수",
    "냉비빔라면": "라면",
    "순한맛 라면": "라면",
    "순한맛라면": "라면",
    "빽라면 만 사용": "라면",
    "곱창라면": "라면",
    "밤 라면": "라면",
    "진라면 순한맛": "라면",

    # 두부/콩류 추가
    "국산콩손두부": "두부",
    "콩비지": "콩",
    "콩물": "콩",
    "콩국수": "콩",
    "통단팥": "팥",

    # 버섯 추가
    "느타리버섯 타래": "버섯",
    "불린표고": "버섯",
    "표고불린물": "버섯",

    # 채소 추가
    "봄동": "배추",
    "알타리무청 줄기": "무",
    "무청": "무",
    "채 썬 무에 꽃소금": "무",
    "달래": "상추",
    "참나물": "상추",
    "쑥갓": "상추",
    "미나리": "상추",
    "봄나물겉절이": "상추",
    "공심채": "상추",
    "청갓": "상추",
    "홍갓": "상추",
    "냉이": "상추",
    "열무": "상추",
    "도라지": "상추",
    "청경채 송이": "상추",
    "부추": "상추",
    "셀러리": "상추",
    "바질": "상추",

    # 감자/전분 추가
    "감자전분": "감자",
    "전분가루": "감자",
    "옥수수전분가루": "감자",

    # 고구마 추가
    "삶은 고구마": "고구마",

    # 토마토 추가
    "토마토소스": "토마토",
    "토마토 주스": "토마토",
    "토마토 페이스트": "토마토",
    "토마토퓌레": "토마토",
    "방울토마토 장아찌": "방울토마토",

    # 오징어/해산물 추가
    "게맛살": "오징어",
    "게살": "오징어",
    "맛살": "오징어",
    "진미채": "오징어",
    "냉동꽃게": "조개",
    "생굴": "조개",
    "골뱅이": "조개",
    "통조림골뱅이": "조개",
    "홍합살": "조개",
    "홍합탕": "조개",
    "홍합밥": "조개",
    "날치": "조기",
    "가다랑어": "마른멸치",
    "디포리": "마른멸치",
    "국멸치": "마른멸치",
    "훈연 멸치": "마른멸치",
    "멸치육수": "마른멸치",
    "훈연멸치육수": "마른멸치",
    "북어채": "명태",
    "황태채": "명태",
    "동태": "명태",
    "동태전": "명태",
    "고등어 통조림": "고등어",
    "고등어통조림": "고등어",
    "손질갈치 토막": "갈치",
    "꽁치 김치찌": "조기",
    "통조림 꽁치": "조기",
    "새우젓": "새우",
    "자숙연근": "새우",

    # 달걀 추가
    "달걀 반죽에 꽃소금": "계란",
    "달걀은 맛소금": "계란",
    "밥양에 따라 달걀": "계란",
    "삶은 메추리": "계란",
    "메추리": "계란",

    # 양파/파 추가
    "만능양파": "양파",
    "채 썬 양배추": "양배추",

    # 밥/쌀 추가
    "밥": "쌀",
    "김밥용 밥": "쌀",
    "밑간 한 밥": "쌀",
    "밑간한 밥": "쌀",
    "밥 밑간": "쌀",
    "밥 밑간하기": "쌀",
    "밥 약 공기": "쌀",
    "밥에 맛소금": "쌀",
    "찬 밥 공기": "쌀",
    "찰옥수수": "쌀",
    "당면": "국수",
    "잡채": "국수",
    "중력분": "밀가루",
    "강력분": "밀가루",
    "빵가루": "밀가루",
    "습식 빵가루": "밀가루",
    "수제비 반죽": "밀가루",
    "튀김가루": "밀가루",
    "베이킹파우더": "밀가루",
    "드라이 이스트": "밀가루",

    # 설탕/단맛 추가
    "단무지 절임용 황설탕": "설탕",
    "카라멜": "설탕",
    "캐러멜": "설탕",
    "캬라멜": "설탕",
    "연유": "설탕",
    "딸기잼": "딸기",
    "사과잼": "사과",

    # 참기름/오일 추가
    "식용유": "참기름",
    "튀김용 기름": "참기름",
    "튀김용 식용유": "참기름",
    "올리브오일": "참기름",
    "엑스트라버진 올리브유": "참기름",
    "마가린": "버터",
    "스틱버터": "버터",

    # 고추장/된장 추가
    "재래식 된장": "된장",
    "재래식된장": "된장",
    "재래식 된": "된장",
    "재래식된": "된장",

    # 소금류
    "꽃소금": "간장",
    "맛소금": "간장",
    "천일염": "간장",
    "소금 g": "간장",

    # 김치류
    "김치": "배추",
    "깍두기": "무",
    "신 김치": "배추",
    "이북식 김치": "배추",
    "절인배추": "배추",

    # 소시지/햄/가공육
    "햄": "돼지고기",
    "소시지": "돼지고기",
    "베이컨": "돼지고기",
    "스팸": "돼지고기",
    "빽햄": "돼지고기",
    "빽소시지": "돼지고기",
    "통조림 햄": "돼지고기",
    "통조림햄": "돼지고기",

    # 치즈/유제품
    "슬라이스치즈": "우유",
    "슬라이스 체다치즈": "우유",
    "슬라이스체다치즈": "우유",
    "체다슬라이스치즈": "우유",
    "체다치즈슬라이스": "우유",
    "체더슬라이스치즈": "우유",
    "모짜렐라 치즈": "우유",
    "모짜렐라슬라이스치즈": "우유",
    "슈레드눈꽃치즈": "우유",
    "피자치즈": "우유",
    "버터": "우유",
    "생크림": "우유",
    "바닐라아이스크림": "우유",
    "고르곤졸라치즈": "우유",
    "그라나파다노치즈": "우유",
    "파르메산 치즈": "우유",
    "파르메산치즈가루": "우유",
    "파마산 치즈": "우유",
    "파마산 치즈가루": "우유",
    "파마산치즈가루": "우유",
    "파르미자노 치즈": "우유",
    "페코리노 치즈": "우유",
    "프로볼로네 치즈": "우유",
    "땅콩버터": "콩",

    # 참치
    "참치": "고등어",
    "통조림참치": "고등어",
    "반죽 참치": "고등어",

    # 어묵
    "어묵": "두부",
    "사각어묵": "두부",
    "구운어묵": "두부",
    "매화어묵": "두부",
    "유부": "두부",

    # 만두
    "만두피": "만두",
    "만두소": "만두",
    "냉동만두 봉지개입": "만두",

    # 김
    "김": "깻잎",
    "김가루": "깻잎",
    "고명용 김": "깻잎",
    "구운 파래김": "깻잎",

    # 미림/청주/술
    "미림": "식초",
    "청주": "식초",
    "막걸리": "식초",
    "소주": "식초",
    "레몬즙": "식초",
    "레몬쥬스": "식초",
    "사과식초": "식초",
    "환만식초": "식초",

    # 케첩/소스류
    "케첩": "케찹",
    "굴소스": "간장",
    "노두유": "간장",
    "조선간장국간": "간장",
    "액젓": "간장",
    "고형 카레": "밀가루",
    "고형카레": "밀가루",
    "춘장": "된장",
}


def get_ingredient_prices(cursor):
    """avg_price 있는 재료의 {name: avg_price} 딕셔너리 반환"""
    cursor.execute("SELECT name, avg_price FROM ingredients WHERE avg_price IS NOT NULL")
    return {row['name']: row['avg_price'] for row in cursor.fetchall()}


def update_mapped_prices(cursor, price_map):
    """매핑 딕셔너리 기반으로 avg_price NULL인 재료 업데이트"""
    updated = 0
    skipped = 0

    for source_name, target_name in PRICE_MAPPING.items():
        # 타겟 재료 가격 확인
        target_price = price_map.get(target_name)
        if not target_price:
            log.warning(f"기준 재료 없음: {target_name}")
            skipped += 1
            continue

        # source_name과 일치하는 재료 업데이트 (avg_price가 NULL인 것만)
        cursor.execute("""
            UPDATE ingredients 
            SET avg_price = %s
            WHERE name = %s AND avg_price IS NULL
        """, (target_price, source_name))

        if cursor.rowcount > 0:
            log.info(f"  {source_name} → {target_name} ({target_price}원)")
            updated += cursor.rowcount

    return updated, skipped


def main():
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor(dictionary=True)
        log.info("DB 연결 성공")
    except Error as e:
        log.error(f"DB 연결 실패: {e}")
        return

    try:
        # 현재 가격 있는 재료 로드
        price_map = get_ingredient_prices(cursor)
        log.info(f"기준 재료 {len(price_map)}개 로드")

        # 매핑 업데이트
        updated, skipped = update_mapped_prices(cursor, price_map)
        conn.commit()

        log.info(f"=== 완료 ===")
        log.info(f"업데이트: {updated}개 / 기준재료 없음: {skipped}개")

        # 결과 확인
        cursor.execute("""
            SELECT 
                COUNT(*) as 전체,
                SUM(CASE WHEN avg_price IS NOT NULL THEN 1 ELSE 0 END) as 가격있음,
                SUM(CASE WHEN avg_price IS NULL THEN 1 ELSE 0 END) as 가격없음
            FROM ingredients
        """)
        row = cursor.fetchone()
        log.info(f"전체: {row['전체']} | 가격있음: {row['가격있음']} | 가격없음: {row['가격없음']}")

    except Error as e:
        log.error(f"DB 오류: {e}")
        conn.rollback()
    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    main()

2026-03-08 17:47:34,184 [INFO] DB 연결 성공


2026-03-08 17:47:34,189 [INFO] 기준 재료 245개 로드
2026-03-08 17:47:34,239 [INFO]   다진생강 → 깐마늘 (1493.00원)
2026-03-08 17:47:34,240 [INFO]   한 생강 → 깐마늘 (1493.00원)
2026-03-08 17:47:34,253 [INFO]   불고기 → 소고기(국산) (15730.00원)
2026-03-08 17:47:34,255 [INFO]   갈비탕 → 소고기(국산) (15730.00원)
2026-03-08 17:47:34,256 [INFO]   소고기 → 소고기(국산) (15730.00원)
2026-03-08 17:47:34,259 [INFO]   탕수육고기 → 돼지고기 (3180.00원)
2026-03-08 17:47:34,260 [INFO]   편육 → 돼지고기 (3180.00원)
2026-03-08 17:47:34,261 [INFO]   족발 → 돼지고기 (3180.00원)
2026-03-08 17:47:34,266 [INFO]   돈등심 → 돼지고기 (3180.00원)
2026-03-08 17:47:34,269 [INFO]   돼지고기뒷다리살 → 돼지고기 (3180.00원)
2026-03-08 17:47:34,270 [INFO]   돼지고기등심 → 돼지고기 (3180.00원)
2026-03-08 17:47:34,272 [INFO]   뒷다리살카레 → 돼지고기 (3180.00원)
2026-03-08 17:47:34,274 [INFO]   돼지등갈비튀김 → 돼지고기 (3180.00원)
2026-03-08 17:47:34,277 [INFO]   치킨 → 닭고기 (1080.00원)
2026-03-08 17:47:34,279 [INFO]   조미닭가슴살 → 닭고기 (1080.00원)
2026-03-08 17:47:34,283 [INFO]   찜닭 → 닭고기 (1080.00원)
2026-03-08 17:47:34,285 [INFO]   스파게티면 → 국수 (394.4